In [5]:
import numpy as np
import cv2
import imutils
import datetime
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import CLIPTokenizer
import onnxruntime as ort

### Gun-type estimates
Install dependencies with `python -m pip install -r requirements.txt`, then run the notebook from top to bottom. The first run downloads the CLIP ONNX model from Hugging Face (about 240 MB). The displayed type and match score are zero-shot estimates from a general-purpose model, not a firearm-specialized or safety-reliable classification.

In [6]:
gun_cascade = cv2.CascadeClassifier("cascade.xml")
if gun_cascade.empty():
    raise FileNotFoundError("Could not load cascade.xml")

MODEL_ID = "Xenova/clip-vit-base-patch32"
GUN_TYPES = ["pistol", "revolver", "rifle", "shotgun", "submachine gun", "non-gun object"]

text_model_path = hf_hub_download(MODEL_ID, "onnx/text_model_quantized.onnx")
vision_model_path = hf_hub_download(MODEL_ID, "onnx/vision_model_quantized.onnx")
text_session = ort.InferenceSession(text_model_path, providers=["CPUExecutionProvider"])
vision_session = ort.InferenceSession(vision_model_path, providers=["CPUExecutionProvider"])
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID)

tokens = tokenizer(
    [f"a photo of a {gun_type}" for gun_type in GUN_TYPES],
    padding="max_length",
    max_length=77,
    truncation=True,
    return_tensors="np",
)
text_features = text_session.run(
    ["text_embeds"], {"input_ids": tokens["input_ids"].astype(np.int64)}
)[0]
text_features /= np.linalg.norm(text_features, axis=-1, keepdims=True)

CLIP_MEAN = np.array([0.48145466, 0.4578275, 0.40821073], dtype=np.float32)
CLIP_STD = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)


def classify_gun_type(frame_roi):
    image = Image.fromarray(cv2.cvtColor(frame_roi, cv2.COLOR_BGR2RGB))
    width, height = image.size
    scale = 224 / min(width, height)
    image = image.resize((round(width * scale), round(height * scale)), Image.Resampling.BICUBIC)
    left = (image.width - 224) // 2
    top = (image.height - 224) // 2
    image = image.crop((left, top, left + 224, top + 224))

    pixels = np.asarray(image, dtype=np.float32) / 255.0
    pixels = ((pixels - CLIP_MEAN) / CLIP_STD).transpose(2, 0, 1)[None, ...]
    image_features = vision_session.run(
        ["image_embeds"], {"pixel_values": pixels}
    )[0]
    image_features /= np.linalg.norm(image_features, axis=-1, keepdims=True)

    logits = (image_features @ text_features.T)[0] * 100
    probabilities = np.exp(logits - logits.max())
    probabilities /= probabilities.sum()
    best = int(np.argmax(probabilities))
    return GUN_TYPES[best], float(probabilities[best])

camera = cv2.VideoCapture(0)
if not camera.isOpened():
    raise RuntimeError("Could not open webcam (camera index 0)")

In [7]:
firstFrame = None
gun_exist = None

In [8]:
try:
    while True:
        ret, frame = camera.read()
        if not ret or frame is None:
            print("Camera se frame nahi mila. Camera connection check karo.")
            break

        frame = imutils.resize(frame, width=500)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        guns = gun_cascade.detectMultiScale(
            gray, scaleFactor=1.3, minNeighbors=5, minSize=(100, 100)
        )

        for (x, y, w, h) in guns:
            roi = frame[y:y + h, x:x + w]
            if roi.size == 0:
                continue

            gun_type, match_score = classify_gun_type(roi)
            label = f"{gun_type} (match {match_score:.0%})"
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
            cv2.putText(
                frame, label, (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2
            )

        cv2.imshow("Security feed", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    camera.release()
    cv2.destroyAllWindows()

KeyboardInterrupt: 